# 31 — FIM × MBL Interaction: Does the Strange Loop Break Localisation?

**UNPRECEDENTED:** Session 1 found that the SCPN XY Hamiltonian shows MBL
protection (Poisson level spacing at n=8). Separately, the strange loop
(FIM) enables large-N sync.

**Critical question:** If FIM delocalises (breaks MBL), it destroys the
very protection that makes the quantum identity stable. If FIM preserves
MBL, identity is DOUBLY protected: localised AND self-observing.

## Method
Add a FIM-analogue term to the XY Hamiltonian:
H_FIM = H_XY + λ · Σ_i (σ_z^i - <σ_z>)²

This is the quantum version of "each oscillator adjusts to match
the collective z-magnetisation" — the quantum strange loop.

Then compute:
1. Level spacing ratio r̄ (Poisson=0.386 localised, GOE=0.530 ergodic)
2. Eigenstate entanglement entropy
3. Participation ratio (localisation in Fock space)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27


def build_xy_hamiltonian(n, K, omega):
    """Build XY Hamiltonian as dense matrix.
    H = -Σ_ij K[i,j] (X_i X_j + Y_i Y_j) - Σ_i ω_i Z_i
    """
    dim = 2**n
    H = np.zeros((dim, dim), dtype=complex)

    # Z terms: -ω_i Z_i
    for i in range(n):
        for state in range(dim):
            bit = (state >> (n - 1 - i)) & 1
            z_val = 1 - 2 * bit  # |0⟩ → +1, |1⟩ → -1
            H[state, state] -= omega[i] * z_val

    # XY terms: -K[i,j] (X_i X_j + Y_i Y_j) = -2K[i,j] flip-flop
    for i in range(n):
        for j in range(i + 1, n):
            if abs(K[i, j]) < 1e-10:
                continue
            for state in range(dim):
                bit_i = (state >> (n - 1 - i)) & 1
                bit_j = (state >> (n - 1 - j)) & 1
                if bit_i != bit_j:  # flip-flop: |01⟩ ↔ |10⟩
                    flipped = state ^ (1 << (n - 1 - i)) ^ (1 << (n - 1 - j))
                    H[state, flipped] -= 2 * K[i, j]

    return H


def add_fim_term(H, n, lam):
    """Add quantum FIM term: λ Σ_i (Z_i - <Z>)²
    This penalises deviation from the mean magnetisation.
    Effectively: λ Σ_i Z_i² - (λ/n)(Σ_i Z_i)²
    First term is identity (Z² = I). Second is the collective term.
    """
    dim = 2**n
    H_fim = H.copy()

    # Compute total magnetisation M = Σ_i Z_i for each basis state
    M = np.zeros(dim)
    for state in range(dim):
        m = 0
        for i in range(n):
            bit = (state >> (n - 1 - i)) & 1
            m += 1 - 2 * bit
        M[state] = m

    # FIM term: -λ/n * (Σ Z_i)² = -λ/n * M² (diagonal)
    # This favours states with large |M| (collective alignment)
    for state in range(dim):
        H_fim[state, state] -= lam * M[state] ** 2 / n

    return H_fim


def level_spacing_ratio(eigvals):
    """Mean adjacent gap ratio r̄.
    Poisson ≈ 0.386, GOE ≈ 0.530, GUE ≈ 0.603"""
    gaps = np.diff(np.sort(eigvals))
    gaps = gaps[gaps > 1e-12]  # remove degeneracies
    if len(gaps) < 2:
        return float("nan")
    ratios = []
    for i in range(len(gaps) - 1):
        r = min(gaps[i], gaps[i + 1]) / max(gaps[i], gaps[i + 1])
        ratios.append(r)
    return float(np.mean(ratios))


def eigenstate_entropy(eigvecs, n):
    """Mean half-chain entanglement entropy of eigenstates."""
    dim = 2**n
    half = n // 2
    dim_A = 2**half
    dim_B = 2 ** (n - half)

    entropies = []
    for col in range(min(dim, 50)):  # sample 50 states
        psi = eigvecs[:, col]
        # Reshape to bipartite
        psi_matrix = psi.reshape(dim_A, dim_B)
        rho_A = psi_matrix @ psi_matrix.conj().T
        eigvals_rho = np.linalg.eigvalsh(rho_A)
        eigvals_rho = eigvals_rho[eigvals_rho > 1e-12]
        S = -np.sum(eigvals_rho * np.log(eigvals_rho))
        entropies.append(float(S))

    return float(np.mean(entropies))


print("Ready.")

In [ ]:
# Sweep FIM strength for different system sizes
results = []
K_SCALE = 1.0

print(f"{'n':>3} {'λ':>6} {'r̄':>8} {'S_ent':>8} {'MBL?':>6}")
for n in [4, 6, 8]:
    K = build_knm_paper27(L=n) * K_SCALE
    omega = OMEGA_N_16[:n]
    H_base = build_xy_hamiltonian(n, K, omega)

    for lam in [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]:
        if lam > 0:
            H = add_fim_term(H_base, n, lam)
        else:
            H = H_base.copy()

        eigvals, eigvecs = np.linalg.eigh(H.real)  # Hermitian
        r_bar = level_spacing_ratio(eigvals)
        S_ent = eigenstate_entropy(eigvecs, n)

        # MBL if r̄ < 0.46 (closer to Poisson 0.386 than GOE 0.530)
        is_mbl = "MBL" if r_bar < 0.46 else "ERGODIC"

        results.append(
            {
                "n": n,
                "lambda": lam,
                "r_bar": round(r_bar, 4),
                "S_ent": round(S_ent, 4),
                "regime": is_mbl,
            }
        )
        print(f"{n:3d} {lam:6.1f} {r_bar:8.4f} {S_ent:8.4f} {is_mbl:>6}")

In [ ]:
# Analysis
print("\n=== FIM × MBL VERDICT ===")
for n in [4, 6, 8]:
    r_no_fim = [r for r in results if r["n"] == n and r["lambda"] == 0][0]["r_bar"]
    r_max_fim = [r for r in results if r["n"] == n and r["lambda"] == 5.0][0]["r_bar"]
    print(f"n={n}: r̄(λ=0) = {r_no_fim:.4f}, r̄(λ=5) = {r_max_fim:.4f}", end="")
    if r_max_fim > r_no_fim + 0.02:
        print("  → FIM DELOCALISES (pushes toward ergodic)")
    elif r_max_fim < r_no_fim - 0.02:
        print("  → FIM ENHANCES MBL (pushes toward Poisson)")
    else:
        print("  → FIM has MINIMAL EFFECT on level statistics")

# Entanglement entropy trend
print("\nEntanglement entropy trend with FIM:")
for n in [4, 6, 8]:
    S_no = [r for r in results if r["n"] == n and r["lambda"] == 0][0]["S_ent"]
    S_max = [r for r in results if r["n"] == n and r["lambda"] == 5.0][0]["S_ent"]
    print(f"n={n}: S(λ=0)={S_no:.4f}, S(λ=5)={S_max:.4f}, change={S_max - S_no:+.4f}")

In [ ]:
# Save
output = {
    "experiment": "FIM x MBL interaction — does strange loop break localisation?",
    "K_scale": K_SCALE,
    "results": results,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/fim_mbl_interaction_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")